In [3]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
cmbs = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/commercial_mortgage.csv')
cmbs.head()

,observation_date,ASCMA
0,2000-01-01,1086111
1,2000-04-01,1116557
2,2000-07-01,1138558
3,2000-10-01,1166539
4,2001-01-01,1147919


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.stats.stattools import durbin_watson

cs = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/DataExport_office_v2.xlsx')
ff = pd.read_excel('../data/FEDFUNDS (1).xlsx')
bond = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/bond_yield_10yr.xlsx')
rrp = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/real_risk_premieum.csv')
inf_e = pd.read_excel('/Users/elyas/vscode/market_analysis_03_office/data/Inflation expectations (1).xlsx')
cmbs = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/commercial_mortgage.csv')
gdp = pd.read_csv('/Users/elyas/vscode/market_analysis_03_office/data/GDP.csv')

#create a subset where column Population is greater than 500000
cs = cs[cs['Population'] > 500000]

#convert to time series
def convert_period_to_datetime(period):
	parts = period.split('Q')
	year = parts[0].strip()
	quarter = parts[1].strip().split()[0]
	month = (int(quarter) - 1) * 3 + 1
	return pd.Timestamp(year=int(year), month=month, day=1)

cs['Period'] = cs['Period'].apply(convert_period_to_datetime)

#remove periods in the year 2025
cs = cs[cs['Period'].dt.year != 2025]

#get data only where geography has Atlanta
#cs = cs[cs['Geography Name'].str.contains('Atlanta')]

#change the column name to period
cmbs.rename(columns={'observation_date': 'period'}, inplace=True)
gdp.rename(columns={'observation_date': 'period'}, inplace=True)
cmbs.rename(columns={'ASCMA': 'oustanding_mortgage'}, inplace=True)
gdp.rename(columns={'GDP': 'gdp'}, inplace=True)
rrp.rename(columns={'observation_date': 'period'}, inplace=True)
inf_e.rename(columns={'Model Output Date': 'period'}, inplace=True)


#the units in outstanding mortgage is in millions, let's convert it to billions
cmbs['oustanding_mortgage'] = cmbs['oustanding_mortgage'] / 1000

#create a new column in cmbs called cmbs_to_gdp
cmbs['cmbs_to_gdp'] = cmbs['oustanding_mortgage'] / gdp['gdp']


#convert fed funds rate to quarterly
ff['period'] = pd.to_datetime(ff['period'])
ff.set_index('period', inplace=True)
ff = ff.resample('Q').mean()
#each data should start on first of the quarter
ff.index = ff.index.to_period('Q').start_time
#change the column period from index to column
ff.reset_index(inplace=True)


#convert real risk premieum to quarterly
inf_e['period'] = pd.to_datetime(inf_e['period'])
inf_e.set_index('period', inplace=True)
inf_e = inf_e.resample('Q').mean()
#change the variable TENEXPCHAREARISPRE to ten_yr_risk_premium
inf_e.rename(columns={'TENEXPCHAREARISPRE': 'ten_yr_risk_premium'}, inplace=True)
#each data should start on first of the quarter
inf_e.index = inf_e.index.to_period('Q').start_time
#change the column period from index to column
inf_e.reset_index(inplace=True)

#convert real risk premieum to quarterly
rrp['period'] = pd.to_datetime(rrp['period'])
rrp.set_index('period', inplace=True)
rrp = rrp.resample('Q').mean()
#change the variable TENEXPCHAREARISPRE to ten_yr_risk_premium
rrp.rename(columns={'TENEXPCHAREARISPRE': 'ten_yr_risk_premium'}, inplace=True)
#each data should start on first of the quarter
rrp.index = rrp.index.to_period('Q').start_time
#change the column period from index to column
rrp.reset_index(inplace=True)

#get data from 2000
inf_e = inf_e[inf_e['period'] >= '2000-01-01']

#Get the data we want

cs = cs[
    [
        "Geography Name",
        "Period",
        "Appreciation Return",
        "Availability Rate",
        "Available SF Direct",
        "Average Sale Price",
        "Cap Rate",
        "Market Cap Rate",
        "Construction Starts SF",
        "Construction Starts SF 12 Mo",
        "Demand SF",
        "Demolished SF",
        "Gross Delivered SF",
        "Inventory SF",
        "Leasing SF Total",
        "Market Asking Rent Growth",
        "Market Asking Rent Growth 12 Mo",
        "Net Absorption SF",
        "Net Absorption SF 12 Mo",
        "Net Delivered SF",
        "Net Delivered SF 12 Mo",
        "Occupancy Rate",
        "Sales Volume Transactions",
        "Sold Building SF",
        "Vacancy Rate",
        "Under Construction SF",
        "Total Sales Transactions"
    ]
].copy()

# Next, rename columns
cs.rename(
    columns={
        "Geography Name": "geography",
        "Period": "period",
        "Appreciation Return": "appreciation_return",
        "Availability Rate": "availability_rate",
        "Available SF Direct": "available_df_direct",
        "Average Sale Price": "avg_sale_price",
        "Cap Rate": "cap_rate",
        "Market Cap Rate": "market_cap_rate",
        "Construction Starts SF": "starts_sf",
        "Construction Starts SF 12 Mo": "starts_sf_12_mo",
        "Demand SF": "demand_sf",
        "Demolished SF": "demolished_sf",
        "Gross Delivered SF": "gross_delivered_sf",
        "Inventory SF": "inventory_sf",
        "Leasing SF Total": "leasing_sf_total",
        "Market Asking Rent Growth": "asking_rent_growth",
        "Market Asking Rent Growth 12 Mo": "asking_rent_growth_12_mo",
        "Net Absorption SF": "net_absorp_sf",
        "Net Absorption SF 12 Mo": "net_absorp_sf_12_mo",
        "Net Delivered SF": "net_delivered_sf",
        "Net Delivered SF 12 Mo": "net_delivered_sf_12_mo",
        "Occupancy Rate": "occupancy_rate",
        "Sales Volume Transactions": "sales_volume",
        "Sold Building SF": "sold_building_sf",
        "Vacancy Rate": "vacancy_rate",
        "Under Construction SF": "under_construction_sf",
        "Total Sales Transactions": "total_sales_transactions"
    },
    inplace=True
)


### merge datasets

cmbs['period'] = pd.to_datetime(cmbs['period'])
cs = pd.merge(cs, bond, on='period', how='left')
cs = pd.merge(cs, inf_e, on='period', how='left')
cs = pd.merge(cs, rrp, on='period', how='left')
cs = pd.merge(cs, ff, on='period', how='left')
cs = pd.merge(cs, cmbs[['period', 'cmbs_to_gdp']], on='period', how='left')
cs = pd.merge(cs, bond, on='period', how='left')


#convert variables to percentage
#multiply by 100 availability rate
cs['availability'] = cs['availability_rate'] * 100
cs['cap_rate'] = cs['cap_rate'] * 100
cs['market_cap_rate'] = cs['market_cap_rate'] * 100
cs['asking_rent_growth'] = cs['asking_rent_growth'] * 100
cs['asking_rent_growth_12_mo'] = cs['asking_rent_growth_12_mo'] * 100
cs['occupancy_rate'] = cs['occupancy_rate'] * 100
cs['vacancy_rate'] = cs['vacancy_rate'] * 100
cs['appreciation_return'] = cs['appreciation_return'] * 100
cs['one_yr_inf_exp'] = cs['one_yr_inf_exp'] * 100
cs['two_yr_inf_exp'] = cs['two_yr_inf_exp'] * 100
cs['three_yr_inf_exp'] = cs['three_yr_inf_exp'] * 100
cs['four_yr_inf_exp'] = cs['four_yr_inf_exp'] * 100
cs['five_yr_inf_exp'] = cs['five_yr_inf_exp'] * 100
cs['six_yr_inf_exp'] = cs['six_yr_inf_exp'] * 100
cs['seven_yr_inf_exp'] = cs['seven_yr_inf_exp'] * 100
cs['eight_yr_inf_exp'] = cs['eight_yr_inf_exp'] * 100
cs['nine_yr_inf_exp'] = cs['nine_yr_inf_exp'] * 100
cs['ten_yr_inf_exp'] = cs['ten_yr_inf_exp'] * 100
cs['cmbs_to_gdp'] = cs['cmbs_to_gdp'] * 100



TypeError: C function scipy.spatial._qhull._barycentric_coordinates has wrong signature (expected void (int, double *, double *, double *), got void (int, double *, double const *, double *))

In [2]:
#export the data to csv
cs.to_csv('../data/office_data_national.csv', index=False)